In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from desdeo.problem.testproblems import summer_cabin_battery_problem
from desdeo.tools import add_epsilon_constraints
from desdeo.tools.gurobipy_solver_interfaces import GurobipySolver

In [ ]:
prob = summer_cabin_battery_problem()
print("Objectives:", [o.symbol for o in prob.objectives])
print("Variables: ", [v.symbol for v in prob.variables])

## Find the ideal point

Minimize `f_1` (electricity cost) with `f_2` (investment cost) effectively unconstrained to find the best achievable electricity cost and the corresponding investment.

In [ ]:
problem_unconstrained, scal_sym, _ = add_epsilon_constraints(
    problem=prob,
    symbol="eps_scal",
    constraint_symbols={"f_2": "eps_f2"},
    objective_symbol="f_1",
    epsilons={"f_2": 1e9},
)

result_ideal = GurobipySolver(problem_unconstrained).solve(scal_sym)

f1_ideal = result_ideal.optimal_objectives["f_1"]
f2_at_f1_ideal = result_ideal.optimal_objectives["f_2"]

print(f"f1 ideal  = {f1_ideal:.2f} EUR")
print(f"f2 at f1* = {f2_at_f1_ideal:.2f} EUR  (battery + solar investment to achieve it)")

## Pareto front sweep

Sweep `epsilon_f2` from 0 (no investment allowed) up to the value at the f1 ideal.
For each epsilon, minimize `f_1` subject to `f_2 ≤ epsilon`.

In [ ]:
N_POINTS = 25
epsilons_f2 = np.linspace(0, f2_at_f1_ideal, N_POINTS)

pareto_f1 = []
pareto_f2 = []

for eps in epsilons_f2:
    problem_eps, scal_sym, _ = add_epsilon_constraints(
        problem=prob,
        symbol="eps_scal",
        constraint_symbols={"f_2": "eps_f2"},
        objective_symbol="f_1",
        epsilons={"f_2": float(eps)},
    )

    result = GurobipySolver(problem_eps).solve(scal_sym)

    if result.success:
        f1 = result.optimal_objectives["f_1"]
        f2 = result.optimal_objectives["f_2"]
        pareto_f1.append(f1)
        pareto_f2.append(f2)
        print(f"eps={eps:7.0f}  →  f1={f1:8.2f} EUR,  f2={f2:7.2f} EUR")
    else:
        print(f"eps={eps:7.0f}  →  FAILED: {result.message}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pareto_f2, pareto_f1, marker="o", color="steelblue")
ax.set_xlabel("Investment cost (EUR)")
ax.set_ylabel("Electricity cost over summer season (EUR)")
ax.set_title("Pareto front: electricity cost vs. battery + solar investment")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()